# Getting Started with ChemiAccel

This notebook provides an interactive introduction to ChemiAccel's core functionality, from basic molecular descriptor calculations to advanced GPU-accelerated processing.

## Prerequisites

Ensure ChemiAccel is installed with the appropriate backend support:

```bash
pip install chemi-accel[cuda]  # For GPU acceleration
```

In [ ]:
# Import required libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from chemi_accel import Calculator

# Set up plotting
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("ChemiAccel Interactive Tutorial")
print("================================")

## 1. Basic Calculator Setup

Let's start by initializing the ChemiAccel calculator and exploring available backends.

In [ ]:
# Initialize calculator with automatic backend selection
calc = Calculator()

print(f"Active backend: {calc.backend.name}")
print(f"Available backends: {calc.available_backends}")
print(f"GPU acceleration: {'Yes' if 'cuda' in calc.available_backends else 'No'}")

## 2. Processing Your First Molecules

Let's calculate basic molecular descriptors for a set of common organic molecules.

In [ ]:
# Define a set of molecules
molecules = {
    "Ethanol": "CCO",
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Benzene": "c1ccccc1",
    "Water": "O",
    "Glucose": "C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O)O)O)O)O",
    "Testosterone": "C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)CCC4=CC(=O)CC[C@]34C",
}

smiles_list = list(molecules.values())
names = list(molecules.keys())

print(f"Processing {len(smiles_list)} molecules...")

# Calculate basic descriptors
results = calc.pandas(smiles_list)
results["name"] = names

# Reorder columns to put name first
cols = ["name"] + [col for col in results.columns if col != "name"]
results = results[cols]

print("\nMolecular Descriptors:")
display(results)

## 3. Visualizing Molecular Properties

Let's create some interactive visualizations to explore the relationships between molecular properties.

In [ ]:
# Create a figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Molecular Property Analysis", fontsize=16, fontweight="bold")

# 1. Molecular Weight vs LogP
axes[0, 0].scatter(
    results["molecular_weight"],
    results["logp"],
    s=100,
    alpha=0.7,
    c=range(len(results)),
    cmap="viridis",
)
axes[0, 0].set_xlabel("Molecular Weight (g/mol)")
axes[0, 0].set_ylabel("LogP")
axes[0, 0].set_title("Lipophilicity vs Size")
axes[0, 0].grid(True, alpha=0.3)

# Add molecule names as annotations
for i, name in enumerate(results["name"]):
    axes[0, 0].annotate(
        name,
        (results["molecular_weight"].iloc[i], results["logp"].iloc[i]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

# 2. TPSA Distribution
axes[0, 1].bar(range(len(results)), results["tpsa"], color="lightcoral", alpha=0.7)
axes[0, 1].set_xlabel("Molecule Index")
axes[0, 1].set_ylabel("TPSA (Ų)")
axes[0, 1].set_title("Topological Polar Surface Area")
axes[0, 1].set_xticks(range(len(results)))
axes[0, 1].set_xticklabels(
    [name[:8] + "..." if len(name) > 8 else name for name in results["name"]],
    rotation=45,
    ha="right",
)

# 3. Hydrogen Bond Donors/Acceptors
x = np.arange(len(results))
width = 0.35
axes[1, 0].bar(x - width / 2, results["num_hbd"], width, label="Donors", alpha=0.7)
axes[1, 0].bar(x + width / 2, results["num_hba"], width, label="Acceptors", alpha=0.7)
axes[1, 0].set_xlabel("Molecule")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("Hydrogen Bonding Capacity")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(
    [name[:6] for name in results["name"]], rotation=45, ha="right"
)
axes[1, 0].legend()

# 4. Rotatable Bonds vs Flexibility
axes[1, 1].scatter(
    results["num_rotatable_bonds"],
    results["molecular_weight"],
    s=results["tpsa"] * 2,
    alpha=0.6,
    c=results["logp"],
    cmap="RdYlBu",
)
axes[1, 1].set_xlabel("Rotatable Bonds")
axes[1, 1].set_ylabel("Molecular Weight (g/mol)")
axes[1, 1].set_title("Flexibility vs Size\n(Color: LogP, Size: TPSA)")
cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
cbar.set_label("LogP")

plt.tight_layout()
plt.show()

print("\nProperty Summary Statistics:")
print(results[["molecular_weight", "logp", "tpsa", "num_hbd", "num_hba"]].describe())

## 4. Drug-Likeness Analysis

Let's analyze how well our molecules conform to drug-likeness rules like Lipinski's Rule of Five.

In [ ]:
def lipinski_analysis(df):
    """Analyze Lipinski's Rule of Five compliance"""

    # Lipinski's criteria
    criteria = {
        "MW ≤ 500": df["molecular_weight"] <= 500,
        "LogP ≤ 5": df["logp"] <= 5,
        "HBD ≤ 5": df["num_hbd"] <= 5,
        "HBA ≤ 10": df["num_hba"] <= 10,
    }

    # Create compliance dataframe
    compliance = pd.DataFrame(criteria)
    compliance.index = df["name"]

    # Count violations
    violations = (~compliance).sum(axis=1)
    compliance["Violations"] = violations
    compliance["Drug-like"] = violations <= 1  # Allow 1 violation

    return compliance


# Perform Lipinski analysis
lipinski_results = lipinski_analysis(results)

print("Lipinski's Rule of Five Analysis:")
print("=" * 40)
display(lipinski_results)

# Visualize compliance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Compliance heatmap
criteria_cols = ["MW ≤ 500", "LogP ≤ 5", "HBD ≤ 5", "HBA ≤ 10"]
sns.heatmap(
    lipinski_results[criteria_cols].astype(int),
    annot=True,
    cmap="RdYlGn",
    cbar_kws={"label": "Compliant"},
    ax=ax1,
)
ax1.set_title("Lipinski Criteria Compliance")
ax1.set_ylabel("Molecules")

# Violation distribution
violation_counts = lipinski_results["Violations"].value_counts().sort_index()
ax2.bar(
    violation_counts.index,
    violation_counts.values,
    color=["green", "yellow", "orange", "red"][: len(violation_counts)],
)
ax2.set_xlabel("Number of Violations")
ax2.set_ylabel("Number of Molecules")
ax2.set_title("Distribution of Lipinski Violations")
ax2.set_xticks(range(max(violation_counts.index) + 1))

plt.tight_layout()
plt.show()

# Summary
drug_like_count = lipinski_results["Drug-like"].sum()
print(f"\nSummary: {drug_like_count}/{len(lipinski_results)} molecules are drug-like")
print(
    f"Drug-like molecules: {', '.join(lipinski_results[lipinski_results['Drug-like']].index)}"
)

## 5. Performance Comparison: CPU vs GPU

Let's compare the performance of different backends for processing larger datasets.

In [ ]:
import time

# Generate a larger dataset for performance testing
test_smiles = [
    "CCO",
    "CCN",
    "CC(=O)O",
    "c1ccccc1",
    "CCc1ccccc1",
    "CC(C)O",
    "CC(C)N",
    "CC(=O)OC",
    "Cc1ccccc1",
    "CCOc1ccccc1",
] * 100  # 1000 molecules total

print(f"Performance test with {len(test_smiles)} molecules")
print("=" * 50)

performance_results = {}

# Test available backends
available_backends = calc.available_backends

for backend_name in available_backends:
    try:
        print(f"\nTesting {backend_name.upper()} backend...")

        # Initialize calculator with specific backend
        test_calc = Calculator(backend=backend_name)

        # Warm-up run
        _ = test_calc.pandas(test_smiles[:10])

        # Timed run
        start_time = time.time()
        results_perf = test_calc.pandas(test_smiles)
        end_time = time.time()

        elapsed_time = end_time - start_time
        throughput = len(test_smiles) / elapsed_time

        performance_results[backend_name] = {
            "time": elapsed_time,
            "throughput": throughput,
        }

        print(f"  Time: {elapsed_time:.2f} seconds")
        print(f"  Throughput: {throughput:.1f} molecules/second")

    except Exception as e:
        print(f"  {backend_name} backend failed: {e}")

# Visualize performance comparison
if len(performance_results) > 1:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    backends = list(performance_results.keys())
    times = [performance_results[b]["time"] for b in backends]
    throughputs = [performance_results[b]["throughput"] for b in backends]

    # Processing time comparison
    bars1 = ax1.bar(
        backends, times, color=["skyblue", "lightcoral", "lightgreen"][: len(backends)]
    )
    ax1.set_ylabel("Processing Time (seconds)")
    ax1.set_title("Processing Time by Backend")
    ax1.grid(True, alpha=0.3)

    # Add value labels on bars
    for bar, time_val in zip(bars1, times):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{time_val:.2f}s",
            ha="center",
            va="bottom",
        )

    # Throughput comparison
    bars2 = ax2.bar(
        backends,
        throughputs,
        color=["skyblue", "lightcoral", "lightgreen"][: len(backends)],
    )
    ax2.set_ylabel("Throughput (molecules/second)")
    ax2.set_title("Throughput by Backend")
    ax2.grid(True, alpha=0.3)

    # Add value labels on bars
    for bar, throughput_val in zip(bars2, throughputs):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{throughput_val:.1f}",
            ha="center",
            va="bottom",
        )

    plt.tight_layout()
    plt.show()

    # Calculate speedup
    if "cpu" in performance_results and "cuda" in performance_results:
        speedup = (
            performance_results["cuda"]["throughput"]
            / performance_results["cpu"]["throughput"]
        )
        print(f"\nGPU Speedup: {speedup:.1f}x faster than CPU")

print("\n" + "=" * 50)
print("Performance test completed!")

## 6. Advanced Descriptor Analysis

Let's explore more advanced molecular descriptors and their applications.

In [ ]:
# Calculate advanced descriptors
advanced_smiles = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",  # Aspirin
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",  # Caffeine
    "CC1=CC=C(C=C1)C2=CC(=NN2C3=CC=C(C=C3)S(=O)(=O)N)C(F)(F)F",  # Celecoxib
    "C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)CCC4=CC(=O)CC[C@]34C",  # Testosterone
]

drug_names = ["Aspirin", "Caffeine", "Celecoxib", "Testosterone"]

print("Calculating advanced molecular descriptors...")

# Calculate comprehensive descriptors
advanced_results = calc.pandas(
    advanced_smiles,
    descriptors=[
        "molecular_weight",
        "logp",
        "tpsa",
        "num_rotatable_bonds",
        "num_aromatic_rings",
        "num_aliphatic_rings",
        "fraction_csp3",
        "num_hbd",
        "num_hba",
        "formal_charge",
    ],
)
advanced_results["name"] = drug_names

# Reorder columns
cols = ["name"] + [col for col in advanced_results.columns if col != "name"]
advanced_results = advanced_results[cols]

display(advanced_results)

# Create a radar chart for drug comparison
from math import pi

# Normalize the data for radar chart
properties = [
    "molecular_weight",
    "logp",
    "tpsa",
    "num_rotatable_bonds",
    "fraction_csp3",
]
norm_data = advanced_results[properties].copy()

# Normalize each property to 0-1 scale
for prop in properties:
    min_val = norm_data[prop].min()
    max_val = norm_data[prop].max()
    if max_val > min_val:
        norm_data[prop] = (norm_data[prop] - min_val) / (max_val - min_val)
    else:
        norm_data[prop] = 0.5

# Create radar chart
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection="polar"))

# Number of properties
N = len(properties)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the circle

# Plot each drug
colors = ["red", "blue", "green", "orange"]
for i, drug in enumerate(drug_names):
    values = norm_data.iloc[i].tolist()
    values += values[:1]  # Complete the circle

    ax.plot(angles, values, "o-", linewidth=2, label=drug, color=colors[i])
    ax.fill(angles, values, alpha=0.25, color=colors[i])

# Add property labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels([prop.replace("_", " ").title() for prop in properties])
ax.set_ylim(0, 1)
ax.set_title(
    "Drug Property Comparison\n(Radar Chart)", size=16, fontweight="bold", pad=20
)
ax.legend(loc="upper right", bbox_to_anchor=(1.2, 1.0))
ax.grid(True)

plt.tight_layout()
plt.show()

print("\nProperty Analysis:")
print(
    f"Most lipophilic: {drug_names[advanced_results['logp'].idxmax()]} (LogP = {advanced_results['logp'].max():.2f})"
)
print(
    f"Most polar: {drug_names[advanced_results['tpsa'].idxmax()]} (TPSA = {advanced_results['tpsa'].max():.1f} Ų)"
)
print(
    f"Most flexible: {drug_names[advanced_results['num_rotatable_bonds'].idxmax()]} ({advanced_results['num_rotatable_bonds'].max()} rotatable bonds)"
)
print(
    f"Highest sp³ character: {drug_names[advanced_results['fraction_csp3'].idxmax()]} ({advanced_results['fraction_csp3'].max():.3f})"
)

## 7. Interactive Molecular Visualization

Let's create an interactive widget for exploring molecular properties.

In [ ]:
from IPython.display import HTML, display
from IPython.widgets import Dropdown, IntSlider, interact


def analyze_molecule(smiles_input):
    """Interactive function to analyze any SMILES input"""

    if not smiles_input or smiles_input.strip() == "":
        print("Please enter a valid SMILES string")
        return

    try:
        # Calculate descriptors
        result = calc.pandas([smiles_input.strip()])

        print(f"Analysis for: {smiles_input}")
        print("=" * 50)

        # Basic properties
        print(f"Molecular Weight: {result['molecular_weight'].iloc[0]:.2f} g/mol")
        print(f"LogP: {result['logp'].iloc[0]:.2f}")
        print(f"TPSA: {result['tpsa'].iloc[0]:.1f} Ų")
        print(f"Rotatable Bonds: {result['num_rotatable_bonds'].iloc[0]}")
        print(f"H-Bond Donors: {result['num_hbd'].iloc[0]}")
        print(f"H-Bond Acceptors: {result['num_hba'].iloc[0]}")

        # Drug-likeness assessment
        mw = result["molecular_weight"].iloc[0]
        logp = result["logp"].iloc[0]
        hbd = result["num_hbd"].iloc[0]
        hba = result["num_hba"].iloc[0]

        violations = 0
        if mw > 500:
            violations += 1
        if logp > 5:
            violations += 1
        if hbd > 5:
            violations += 1
        if hba > 10:
            violations += 1

        print(f"\nLipinski Violations: {violations}")
        if violations <= 1:
            print("✅ Drug-like molecule")
        else:
            print("❌ Not drug-like")

        # Create a simple bar chart
        fig, ax = plt.subplots(1, 1, figsize=(8, 4))
        properties = ["MW/100", "LogP", "TPSA/50", "RotBonds", "HBD", "HBA"]
        values = [
            mw / 100,
            logp,
            result["tpsa"].iloc[0] / 50,
            result["num_rotatable_bonds"].iloc[0],
            hbd,
            hba,
        ]

        bars = ax.bar(
            properties,
            values,
            color=["skyblue", "lightcoral", "lightgreen", "gold", "plum", "orange"],
        )
        ax.set_title(f"Molecular Properties: {smiles_input}")
        ax.set_ylabel("Scaled Values")
        plt.xticks(rotation=45)

        # Add value labels on bars
        for bar, value in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{value:.1f}",
                ha="center",
                va="bottom",
            )

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error analyzing molecule: {e}")
        print("Please check that the SMILES string is valid")


# Create interactive input
print("Interactive Molecular Analyzer")
print("Enter a SMILES string to analyze:")

# Example molecules for quick testing
example_molecules = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Penicillin": "CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C",
    "Morphine": "CN1CC[C@]23C4=C5C=CC(=O)C=C5CC[C@H]2[C@H]1C[C@H]3C4=O",
    "Custom": "",  # For manual input
}


def interactive_analyzer(molecule_choice, custom_smiles=""):
    if molecule_choice == "Custom":
        if custom_smiles:
            analyze_molecule(custom_smiles)
        else:
            print("Please enter a SMILES string in the custom field")
    else:
        analyze_molecule(example_molecules[molecule_choice])


# Create the interactive widget
interact(
    interactive_analyzer,
    molecule_choice=Dropdown(
        options=list(example_molecules.keys()), value="Aspirin", description="Molecule:"
    ),
    custom_smiles="",
);

## 8. Next Steps

Congratulations! You've completed the ChemiAccel getting started tutorial. Here are some next steps to explore:

### Advanced Features
- **3D Descriptors**: Calculate conformer-dependent properties
- **Fingerprints**: Generate molecular fingerprints for similarity analysis
- **Batch Processing**: Process large datasets efficiently
- **Cloud Integration**: Scale to cloud platforms

### Domain-Specific Applications
- **Drug Discovery**: ADMET prediction and drug-likeness analysis
- **Polymer Science**: Polymer property prediction and design
- **Materials Science**: Crystal structure analysis and property prediction
- **AI Workflows**: Inverse design and active learning

### Resources
- 📖 [Full Documentation](https://chemi-accel.readthedocs.io)
- 🎯 [Advanced Examples](../examples/)
- 📊 [Domain Tutorials](../domains/)
- ☁️ [Cloud Deployment Guide](../user_guide/cloud_deployment.rst)
- 🖥️ [HPC Usage Guide](../user_guide/hpc_usage.rst)

### Support
- 💬 [Community Forum](https://github.com/chemi-accel/chemi-accel/discussions)
- 🐛 [Issue Tracker](https://github.com/chemi-accel/chemi-accel/issues)
- 📧 [Contact Support](mailto:support@chemi-accel.org)

In [ ]:
# Final summary
print("🎉 Tutorial Complete!")
print("\nIn this notebook, you learned how to:")
print("✅ Initialize ChemiAccel with different backends")
print("✅ Calculate molecular descriptors")
print("✅ Visualize molecular properties")
print("✅ Analyze drug-likeness")
print("✅ Compare backend performance")
print("✅ Use interactive analysis tools")
print("\nReady to accelerate your chemical discovery! 🚀")